In [4]:
import torch_pruning 

import torch
import torch.nn as nn
import random

class RandomMLPNet(nn.Module):
    def __init__(self, input_dim, output_dim, num_mlps=3,
                 hidden_dim_range=(4, 4), num_layers_range=(1, 4)):
        """
        input_dim:  輸入維度
        output_dim: 輸出維度
        num_mlps:   一共要多少個獨立的 MLP 分支
        hidden_dim_range: 隨機隱藏層維度的範圍 (min, max)
        num_layers_range: 隨機隱藏層數的範圍 (min, max)
        """
        super().__init__()
        self.mlps = nn.ModuleList()
        
        for i in range(num_mlps):
            # 隨機決定此 MLP 的隱藏層數量和維度
            num_layers = random.randint(*num_layers_range)
            dims = [input_dim] + [
                random.randint(*hidden_dim_range) for _ in range(num_layers)
            ] + [output_dim]
            
            layers = []
            for j in range(len(dims)-1):
                layers.append(nn.Linear(dims[j], dims[j+1]))
                # 最後一層不加激
            mlp = nn.Sequential(*layers)
            self.mlps.append(mlp)
        
        # 將多個 MLP 的輸出平均
        self.combine = lambda outs: torch.stack(outs, dim=0).mean(dim=0)
    
    def forward(self, x):
        # 每個 MLP 分支分別計算
        outputs = [mlp(x) for mlp in self.mlps]
        # 合併
        return self.combine(outputs)

model = RandomMLPNet(input_dim=4, output_dim=4, num_mlps=2)
example_inputs = torch.randn(1, 4)   # batch_size=16

In [ ]:
import torch
from torchvision.models import resnet18
import torch_pruning as tp


imp = tp.importance.GroupMagnitudeImportance(p=2) 

# 2. Initialize a pruner with the model and the importance criterion
ignored_layers = []
for m in model.modules():
    if isinstance(m, torch.nn.Linear) and m.out_features == 1000:
        ignored_layers.append(m) # DO NOT prune the final classifier!

pruner = tp.pruner.BasePruner( # We can always choose BasePruner if sparse training is not required.
    model,
    example_inputs,
    importance=imp,
    pruning_ratio=0.5, # remove 50% channels, ResNet18 = {64, 128, 256, 512} => ResNet18_Half = {32, 64, 128, 256}
    # pruning_ratio_dict = {model.conv1: 0.2, model.layer2: 0.8}, # customized pruning ratios for layers or blocks
    ignored_layers=ignored_layers,
    round_to=8, # It's recommended to round dims/channels to 4x or 8x for acceleration. Please see: https://docs.nvidia.com/deeplearning/performance/dl-performance-convolutional/index.html
)

base_macs, base_nparams = tp.utils.count_ops_and_params(model, example_inputs)
tp.utils.print_tool.before_pruning(model) # or print(model) 
pruner.step() 
tp.utils.print_tool.after_pruning(model) # or print(model), this util will show the difference before and after pruning 
macs, nparams = tp.utils.count_ops_and_params(model, example_inputs) 
print(f"MACs: {base_macs/1e9} G -> {macs/1e9} G, #Params: {base_nparams/1e6} M -> {nparams/1e6} M") 

RandomMLPNet(
  (mlps): ModuleList(
    (0): Sequential(
      (0): Linear(in_features=4, out_features=4, bias=True)
      (1): Linear(in_features=4, out_features=4, bias=True)
      (2): Linear(in_features=4, out_features=4, bias=True)
      (3): Linear(in_features=4, out_features=4, bias=True)
    )
    (1): Sequential(
      (0): Linear(in_features=4, out_features=4, bias=True)
      (1): Linear(in_features=4, out_features=4, bias=True)
      (2): Linear(in_features=4, out_features=4, bias=True)
    )
  )
)

MACs: 1.4e-07 G -> 1.4e-07 G, #Params: 0.00014 M -> 0.00014 M


In [6]:
for group in pruner.DG.get_all_groups(): 
    print(group._group) 

[(prune_out_channels on mlps.0.3 (Linear(in_features=4, out_features=4, bias=True)) => prune_out_channels on mlps.0.3 (Linear(in_features=4, out_features=4, bias=True)), [0, 1, 2, 3]), (prune_out_channels on mlps.0.3 (Linear(in_features=4, out_features=4, bias=True)) => prune_out_channels on _ElementWiseOp_1(StackBackward0), [0, 1, 2, 3]), (prune_out_channels on _ElementWiseOp_1(StackBackward0) => prune_out_channels on mlps.1.2 (Linear(in_features=4, out_features=4, bias=True)), [0, 1, 2, 3]), (prune_out_channels on _ElementWiseOp_1(StackBackward0) => prune_out_channels on _ElementWiseOp_0(MeanBackward1), [0, 1, 2, 3])]
[(prune_out_channels on mlps.1.1 (Linear(in_features=4, out_features=4, bias=True)) => prune_out_channels on mlps.1.1 (Linear(in_features=4, out_features=4, bias=True)), [0, 1, 2, 3]), (prune_out_channels on mlps.1.1 (Linear(in_features=4, out_features=4, bias=True)) => prune_out_channels on _Reshape_3(), [0, 1, 2, 3]), (prune_out_channels on _Reshape_3() => prune_out_c